In [1]:
from google.colab import userdata
database_url = userdata.get('Analyst_NeonDB_Postgresql_server')

In [2]:
!pip install -q "psycopg[binary]" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 13.4 MB/s eta 0:00:00


In [3]:
import psycopg
conn = psycopg.connect(database_url)
print("Connected to Neon successfully.")

Connected to Neon successfully.


In [4]:
with conn.cursor() as cur:
  cur.execute("SELECT version();")
  print(cur.fetchone()[0])

PostgreSQL 18.6 (c5250a2) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [5]:
from sqlalchemy import create_engine
engine = create_engine(database_url)

In [6]:
import pandas as pd

In [13]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)
tables

,table_name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category
7,products
8,sellers


## Product & Category Analysis

In [9]:
# product/category coverage
query = """
SELECT
    COUNT(*) AS total_products,
    COUNT(product_category_name_english) AS products_with_category,
    COUNT(*) - COUNT(product_category_name_english) AS products_without_category
FROM products;
"""

product_category_coverage = pd.read_sql(query, engine)
product_category_coverage

,total_products,products_with_category,products_without_category
0,32328,32328,0


In [25]:
# Revenue by product category
query = """
SELECT
    COALESCE(
        pc.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category,

    SUM(oi.price) AS product_sales,
    COUNT(DISTINCT oi.order_id) AS orders,
    COUNT(*) AS items_sold,
    AVG(oi.price) AS average_item_price

FROM order_items AS oi

JOIN products AS p
    ON oi.product_id = p.product_id

LEFT JOIN product_category AS pc
    ON p.product_category_name = pc.product_category_name

GROUP BY 1

ORDER BY product_sales DESC;
"""

category_sales = pd.read_sql(query, engine)
category_sales

,category,product_sales,orders,items_sold,average_item_price
0,health_beauty,1258681.34,8836,9670,130.163531
1,watches_gifts,1205005.68,5624,5991,201.135984
2,bed_bath_table,1036988.68,9417,11115,93.296327
3,sports_leisure,988048.97,7720,8641,114.344285
4,computers_accessories,911954.32,6689,7827,116.513903
...,...,...,...,...,...
66,flowers,1110.04,29,33,33.637576
67,home_comfort_2,760.27,24,30,25.342333
68,cds_dvds_musicals,730.00,12,14,52.142857
69,fashion_childrens_clothes,569.85,8,8,71.231250


In [27]:
# Category contribution %
query = """
WITH category_sales AS (
    SELECT
        COALESCE(
            pc.product_category_name_english,
            p.product_category_name,
            'Unknown'
        ) AS category,
        SUM(oi.price) AS product_sales
    FROM order_items AS oi
    JOIN products AS p
        ON oi.product_id = p.product_id
    LEFT JOIN product_category AS pc
        ON p.product_category_name = pc.product_category_name
    GROUP BY 1
)

SELECT
    category,
    product_sales,

    ROUND(
        (100.0 * product_sales / SUM(product_sales) OVER ())::NUMERIC,
        3
    ) AS sales_share_pct

FROM category_sales

ORDER BY product_sales DESC;
"""

category_contribution = pd.read_sql(query, engine)

category_contribution

,category,product_sales,sales_share_pct
0,health_beauty,1258681.34,9.39
1,watches_gifts,1205005.68,8.99
2,bed_bath_table,1036988.68,7.73
3,sports_leisure,988048.97,7.37
4,computers_accessories,911954.32,6.80
...,...,...,...
66,flowers,1110.04,0.01
67,home_comfort_2,760.27,0.01
68,cds_dvds_musicals,730.00,0.01
69,fashion_childrens_clothes,569.85,0.00


In [34]:
import plotly.express as px

# Sort the DataFrame by sales_share_pct for better visualization
pie_data = category_contribution.sort_values(by='sales_share_pct', ascending=False).head(10) # Display top 10 categories for clarity

# Create the interactive pie chart using Plotly
fig = px.pie(pie_data, values='sales_share_pct', names='category',
             title='Top 10 Product Category Sales Share Percentage',
             hover_name='category', hover_data={'sales_share_pct': ':.2f%'})

fig.show()

Pareto analysis is a decision-making technique that uses the 80/20 rule to identify the small number of causes that drive most of a problem or result.

In [39]:
# Pareto analysis
query = """
WITH category_sales AS (
    SELECT
        COALESCE(
            pc.product_category_name_english,
            p.product_category_name,
            'Unknown'
        ) AS category,
        SUM(oi.price) AS product_sales
    FROM order_items AS oi
    JOIN products AS p
        ON oi.product_id = p.product_id
    LEFT JOIN product_category AS pc
        ON p.product_category_name = pc.product_category_name
    GROUP BY 1
),

ranked_categories AS (
    SELECT
        category,
        product_sales,

        SUM(product_sales) OVER (
            ORDER BY product_sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS cumulative_sales

    FROM category_sales
)

SELECT
    category,
    product_sales,
    ROUND(
        (100.0 * cumulative_sales
        / SUM(product_sales) OVER ())::NUMERIC,
        2
    ) AS cumulative_sales_pct

FROM ranked_categories

ORDER BY product_sales DESC;
"""

pareto_categories = pd.read_sql(query, engine)

pareto_categories

,category,product_sales,cumulative_sales_pct
0,health_beauty,1258681.34,9.39
1,watches_gifts,1205005.68,18.38
2,bed_bath_table,1036988.68,26.11
3,sports_leisure,988048.97,33.48
4,computers_accessories,911954.32,40.28
...,...,...,...
66,flowers,1110.04,99.98
67,home_comfort_2,760.27,99.99
68,cds_dvds_musicals,730.00,99.99
69,fashion_childrens_clothes,569.85,100.00


In [44]:
import plotly.express as px

# Create an index for the x-axis to represent the category rank
pareto_categories['category_rank'] = pareto_categories.index + 1

# Create the interactive line chart using Plotly Express
fig = px.line(pareto_categories,
              y='category_rank',
              x='cumulative_sales_pct',
              hover_name='category',
              hover_data={
                  'category_rank': False, # Hide rank from hover
                  'product_sales': ':.2f', # Format product sales
                  'cumulative_sales_pct': ':.2f%' # Format cumulative sales percentage
              },
              title='Pareto Chart: Cumulative Sales Percentage by Product Category Rank',
              labels={'category_rank': 'Category Rank', 'cumulative_sales_pct': 'Cumulative Sales Percentage'})

# Add a line at 80% to highlight the Pareto principle
fig.add_vline(x=80, line_dash="dot", line_color="red", annotation_text="80% Cumulative Sales")

fig.update_traces(mode='lines+markers', marker=dict(size=8, opacity=0.8))
fig.show()

In [47]:
#  Top products
query = """
SELECT
    product_id,
    COUNT(*) AS items_sold,
    COUNT(DISTINCT order_id) AS orders,
    SUM(price) AS product_sales,
    AVG(price) AS average_item_price

FROM order_items
GROUP BY product_id
ORDER BY product_sales DESC;
"""

top_products = pd.read_sql(query, engine)

top_products

,product_id,items_sold,orders,product_sales,average_item_price
0,bb50f2e236e5eea0100680137654686c,195,187,63885.00,327.615385
1,6cdd53843498f92890544667809f1595,156,151,54730.20,350.834615
2,d6160fb7873f184099d9bc95e30376af,35,35,48899.34,1397.124000
3,d1c427060a0f73f6b889a5c7c61f2ac4,343,323,47214.51,137.651633
4,99a4788cb24856965c36a24e339b6058,488,467,43025.56,88.167131
...,...,...,...,...,...
32946,2e8316b31db34314f393806fd7b6e185,1,1,2.99,2.990000
32947,680cc8535be7cc69544238c1d6a83fe8,1,1,2.90,2.900000
32948,8a3254bee785a526d548a81a9bc3c9be,3,3,2.55,0.850000
32949,310dc32058903b6416c71faff132df9e,1,1,2.29,2.290000


In [52]:
# Volume vs Price
query = """
SELECT
    product_id,
    COUNT(*) AS items_sold,
    SUM(price) AS product_sales,
    AVG(price) AS average_price

FROM order_items

GROUP BY product_id

ORDER BY items_sold DESC;
"""
top_volume_products = pd.read_sql(query, engine)

query = """
SELECT
    product_id,
    COUNT(*) AS items_sold,
    SUM(price) AS product_sales,
    AVG(price) AS average_price

FROM order_items

GROUP BY product_id

ORDER BY product_sales DESC;
"""
top_price_products = pd.read_sql(query, engine)


display(top_volume_products, top_price_products)
print("We can compare above tables to find highest selling and highest revenue producing products")

,product_id,items_sold,product_sales,average_price
0,aca2eb7d00ea1a7b8ebd4e68314663af,527,37608.90,71.364137
1,99a4788cb24856965c36a24e339b6058,488,43025.56,88.167131
2,422879e10f46682990de24d770e7f83d,484,26577.22,54.911612
3,389d119b48cf3043d311335e499d9c6b,392,21440.59,54.695383
4,368c6c730842d78016ad823897a372db,388,21056.80,54.270103
...,...,...,...,...
32946,9eb0c62c8c4159426b700eb10045dc44,1,84.90,84.900000
32947,39c4b45f85d57630198b4da1688c13a0,1,19.99,19.990000
32948,580af1ebaa84e22ae14e08f073d091b5,1,19.00,19.000000
32949,cf30110b1e85017c00752838fec35442,1,29.75,29.750000


,product_id,items_sold,product_sales,average_price
0,bb50f2e236e5eea0100680137654686c,195,63885.00,327.615385
1,6cdd53843498f92890544667809f1595,156,54730.20,350.834615
2,d6160fb7873f184099d9bc95e30376af,35,48899.34,1397.124000
3,d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51,137.651633
4,99a4788cb24856965c36a24e339b6058,488,43025.56,88.167131
...,...,...,...,...
32946,2e8316b31db34314f393806fd7b6e185,1,2.99,2.990000
32947,680cc8535be7cc69544238c1d6a83fe8,1,2.90,2.900000
32948,8a3254bee785a526d548a81a9bc3c9be,3,2.55,0.850000
32949,310dc32058903b6416c71faff132df9e,1,2.29,2.290000


We can compare above tables to find highest selling and highest revenue producing products


## Geographic Analysis

In [57]:
# Sales by state
query = """
WITH state_metrics AS (
    SELECT
        c.customer_state,
        COUNT(DISTINCT o.order_id) AS orders,
        SUM(oi.price) AS product_sales,
        SUM(oi.price) / COUNT(DISTINCT o.order_id) AS average_order_value
    FROM orders AS o
    JOIN customers AS c
        ON o.customer_id = c.customer_id
    JOIN order_items AS oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_state
)

SELECT
    customer_state,
    orders,
    product_sales,
    average_order_value,
    ROUND(
        (100.0 * product_sales
        / SUM(product_sales) OVER ())::NUMERIC,
        2
    ) AS sales_share_pct
FROM state_metrics
ORDER BY product_sales DESC;
"""

state_contribution = pd.read_sql(query, engine)
state_contribution

,customer_state,orders,product_sales,average_order_value,sales_share_pct
0,SP,41375,5.202955e+06,125.751179,38.28
1,RJ,12762,1.824093e+06,142.931568,13.42
2,MG,11544,1.585308e+06,137.327445,11.66
3,RS,5432,7.503040e+05,138.126661,5.52
4,PR,4998,6.830838e+05,136.671421,5.03
5,SC,3612,5.205533e+05,144.117757,3.83
6,BA,3358,5.113500e+05,152.278139,3.76
7,DF,2125,3.026039e+05,142.401854,2.23
8,GO,2007,2.945919e+05,146.782237,2.17
9,ES,2025,2.750373e+05,135.820894,2.02


In [56]:
# Sales by order status
query = """
SELECT
    o.order_status,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(oi.price) AS product_sales,
    SUM(oi.freight_value) AS freight_value

FROM orders AS o

LEFT JOIN order_items AS oi
    ON o.order_id = oi.order_id

GROUP BY o.order_status

ORDER BY product_sales DESC;
"""

sales_by_status = pd.read_sql(query, engine)

sales_by_status

,order_status,orders,product_sales,freight_value
0,created,5,NaN,NaN
1,delivered,96478,1.322150e+07,2198275.64
2,shipped,1107,1.507274e+05,26401.90
3,canceled,625,9.523527e+04,10650.45
4,invoiced,314,6.152637e+04,7462.38
5,processing,301,6.043922e+04,8954.89
6,unavailable,609,2.007690e+03,132.80
7,approved,2,2.096000e+02,31.48


In [58]:
# Cancellation rate
query = """
SELECT
    COUNT(*) AS total_orders,

    COUNT(*) FILTER (
        WHERE order_status = 'canceled'
    ) AS canceled_orders,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE order_status = 'canceled'
        )
        / COUNT(*),
        2
    ) AS cancellation_rate_pct

FROM orders;
"""

cancellation = pd.read_sql(query, engine)
cancellation

,total_orders,canceled_orders,cancellation_rate_pct
0,99441,625,0.63
